# Implicit stepping with SOLVAX

Backward Euler takes larger stable steps than RK4 by solving a nonlinear
system each step. MHX forms the residual and a spectral preconditioner,
and [SOLVAX](../physics/solvax_boundary.md) runs the Newton--Krylov solve.
This tutorial compares both integrators on one problem. The committed
outputs ran on a laptop CPU in under twenty seconds.

## Float64 first

The Newton solve targets tolerances near $10^{-9}$, which float32 cannot
reach. Under the JAX default precision the convergence flag therefore
reports failure on every step. Enable float64 before importing MHX for any
implicit run.

In [1]:
import jax

jax.config.update("jax_enable_x64", True)

import numpy as np

import mhx


def make_simulation(integrator, dt):
    return mhx.Simulation(
        shape=(32, 32),
        equilibrium=mhx.CosineTearingEquilibrium(perturbation_amplitude=1.0e-3),
        resistivity=5.0e-3,
        viscosity=5.0e-3,
        dt=dt,
        t_end=1.0,
        save_every=1_000_000,
        integrator=integrator,
        verbose=False,
    )


explicit = make_simulation("rk4", 2.0e-2).run()
implicit = make_simulation("backward_euler", 1.0e-1).run()

print(f"rk4: {explicit.trajectory.times.shape[0]} saved states, "
      f"run {explicit.run_seconds:.2f} s")
print(f"backward euler: run {implicit.run_seconds:.2f} s")

rk4: 1 saved states, run 0.01 s
backward euler: run 0.02 s


## Check convergence before anything else

An implicit run is only evidence when every Newton and GMRES solve
converged. MHX stores the flags in the diagnostics.

In [2]:
print("newton converged:", implicit.diagnostics["implicit_converged"])
print("gmres converged:", implicit.diagnostics["implicit_linear_converged"])

newton converged: True
gmres converged: True


In [3]:
psi_explicit = np.asarray(explicit.final_state.psi)
psi_implicit = np.asarray(implicit.final_state.psi)
difference = np.linalg.norm(psi_implicit - psi_explicit)
reference = np.linalg.norm(psi_explicit)
print(f"relative L2 difference at t = 1: {difference / reference:.3e}")

relative L2 difference at t = 1: 2.025e-06


The implicit run took five times fewer steps and lands close to the RK4
reference. The gap is the first-order time error of backward Euler, and it
shrinks linearly with the step.

Backward Euler pays off when diffusion, not accuracy, limits the explicit
step: strong resistivity, hyper-resistivity, or fine grids. The
[time integration page](../physics/time_integration.md) gives the step
limits, and the implicit path stays differentiable through the SOLVAX
implicit-differentiation rules.